In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# =========================
# 1. Imports
# =========================
import os, re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader

# =========================
# 2. Load Dataset
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/devign-dataset26"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        path = os.path.join(root, f)
        if f.endswith(".csv"):
            df = pd.read_csv(path); break
        elif f.endswith(".json"):
            df = pd.read_json(path); break
    if df is not None:
        break

print("Columns:", df.columns)

# =========================
# 3. Column Mapping
# =========================
df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']

# =========================
# 4. Train-Val-Test Split
# =========================
X_temp, X_test, y_temp, y_test = train_test_split(
    df['text'].astype(str).tolist(), df['label'].values,
    test_size=0.1, random_state=42, stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.111, random_state=42, stratify=y_temp
)

# =========================
# 5. Tokenization
# =========================
def tokenize(text):
    text = text.lower()
    text = re.sub(r'([(){}\[\];,<>!=&|^~])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.split()

counter = Counter()
for text in X_train:
    counter.update(tokenize(text))

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.most_common(30000):
    if freq >= 2:
        vocab[word] = len(vocab)

def encode(text):
    return [vocab.get(w, 1) for w in tokenize(text)]

MAX_LEN = 512

def pad(seq):
    seq = seq[:MAX_LEN]
    return seq + [0] * (MAX_LEN - len(seq))

X_train_enc = [pad(encode(t)) for t in X_train]
X_val_enc   = [pad(encode(t)) for t in X_val]
X_test_enc  = [pad(encode(t)) for t in X_test]

# =========================
# 6. Dataset & Loaders
# =========================
class DevignDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(DevignDataset(X_train_enc, y_train), batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(DevignDataset(X_val_enc,   y_val),   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(DevignDataset(X_test_enc,  y_test),  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# =========================
# 7. Attention Pooling
# =========================
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states):
        scores = self.attn(hidden_states).squeeze(-1)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1)

# =========================
# 8. BiLSTM + Attention Model
# =========================
class BiLSTMAttnModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.emb_norm  = nn.LayerNorm(embed_dim)

        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        lstm_out_dim = hidden_dim * 2
        self.attn_pool = AttentionPooling(lstm_out_dim)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.emb_norm(self.embedding(x))
        lstm_out, _ = self.lstm(x)
        pooled = self.attn_pool(lstm_out)
        return self.classifier(pooled).squeeze()

# =========================
# 9. Setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = BiLSTMAttnModel(len(vocab)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
pos_weight = torch.tensor(class_weights[1] / class_weights[0]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# verbose removed — incompatible with PyTorch >= 2.4
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# =========================
# 10. Training with Early Stopping
# =========================
EPOCHS = 15
PATIENCE = 4
CLIP_GRAD = 1.0

best_val_loss = float('inf')
no_improve = 0
best_model_path = "best_model.pt"

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()
        train_loss += loss.item()

    # --- Validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            val_loss += criterion(model(X_batch), y_batch).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    current_lr  = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

    prev_lr = current_lr
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < prev_lr:
        print(f"           ↓ LR reduced: {prev_lr:.2e} → {new_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"           ✓ Saved best model (val_loss={val_loss:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# =========================
# 11. Evaluation (best checkpoint)
# =========================
model.load_state_dict(torch.load(best_model_path))
model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = torch.sigmoid(model(X_batch.to(device)))
        preds.extend((outputs > 0.5).cpu().numpy())
        true_labels.extend(y_batch.numpy())

print("\nAccuracy:", accuracy_score(true_labels, preds))
print("\nClassification Report:\n", classification_report(true_labels, preds))

Columns: Index(['code', 'label'], dtype='object')
Using device: cuda
Parameters: 2,664,450
Epoch  1 | Train Loss: 0.7857 | Val Loss: 0.7877 | LR: 1.00e-03
           ✓ Saved best model (val_loss=0.7877)
Epoch  2 | Train Loss: 0.7841 | Val Loss: 0.7888 | LR: 1.00e-03
Epoch  3 | Train Loss: 0.7841 | Val Loss: 0.7932 | LR: 1.00e-03
Epoch  4 | Train Loss: 0.7827 | Val Loss: 0.7868 | LR: 1.00e-03
           ✓ Saved best model (val_loss=0.7868)
Epoch  5 | Train Loss: 0.7809 | Val Loss: 0.7909 | LR: 1.00e-03
Epoch  6 | Train Loss: 0.7766 | Val Loss: 0.7863 | LR: 1.00e-03
           ✓ Saved best model (val_loss=0.7863)
Epoch  7 | Train Loss: 0.7760 | Val Loss: 0.7973 | LR: 1.00e-03
Epoch  8 | Train Loss: 0.7649 | Val Loss: 0.7935 | LR: 1.00e-03
Epoch  9 | Train Loss: 0.7463 | Val Loss: 0.8295 | LR: 1.00e-03
           ↓ LR reduced: 1.00e-03 → 5.00e-04
Epoch 10 | Train Loss: 0.7275 | Val Loss: 0.8084 | LR: 5.00e-04
Early stopping triggered at epoch 10

Accuracy: 0.48905109489051096

Classificat